# 🦟 VectorIA
## Módulo 2 · RNN/LSTM: Predicción de casos de dengue
### Notebook 03 — Entrenamiento, evaluación y pronóstico 2025–2026

**Proyecto:** VectorIA — Asistente Inteligente para la Vigilancia del Dengue en Costa Rica
**Archivo de datos:** `vectoria_dengue_nacional_real.csv` (574 semanas reales, 2014–2024)
**Responsable:** Persona 2 (LSTM)

---

### Objetivo
Predecir los **casos de dengue de las próximas 4 semanas** a partir de 26 semanas de historial (casos, lluvia, temperatura máxima y mínima, humedad y estacionalidad), comparando:

* **SimpleRNN** (modelo base) → `SimpleRNN(32) → Dense(4)`
* **LSTM** (modelo principal) → `LSTM(64) → LSTM(32) → Dropout(0.2) → Dense(4)`

Ambos se evalúan con **RMSE** y **MAE** frente al **baseline estacional** del profesor (misma semana del año anterior) y, como referencia adicional, frente a un baseline de **persistencia** (repetir la última semana observada).

Finalmente se generan los **pronósticos para 2025 y 2026**.

> La lógica vive en `src/train/lstm.py`; este notebook la ejecuta y documenta los resultados.

## 1. Importación de librerías y configuración


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Localiza la raíz del proyecto (la carpeta que contiene src/) para poder importar el módulo
RAIZ = Path.cwd()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.train import lstm as L


pd.set_option("display.float_format", "{:.2f}".format)
EPOCAS = 200   # máximo de épocas (EarlyStopping detiene antes)
print("Raíz del proyecto:", RAIZ)
print("TensorFlow:", __import__("tensorflow").__version__)
L.RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

Raíz del proyecto: C:\Users\Eladrel\Desktop\VectorIA
TensorFlow: 2.22.0-rc0


AttributeError: module 'src.train.lstm' has no attribute 'RUTA_FIGURAS'

In [ ]:
from pathlib import Path

# Definir la ruta localmente en el notebook
L.RUTA_FIGURAS = Path("reports/figures")

# Ahora puedes ejecutar tu línea sin que falle
L.RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)



## 2. Carga de datos

Se carga el CSV real y se ordena cronológicamente. Los datos originales no se modifican.

In [ ]:
import importlib

importlib.reload(L)
print(L.__file__)
print([n for n in dir(L) if not n.startswith("_")][:10])
df = L.cargar_datos()
print(f"Filas: {len(df)} | Desde {df['fecha_inicio_semana'].min().date()} hasta {df['fecha_inicio_semana'].max().date()}")
display(df.tail(3))

## 3. Ingeniería de variables (features)

Siguiendo el EDA (`06_EDA_Dengue_Clima.ipynb`):

| Variable | Justificación |
|---|---|
| `log_casos` = log(1 + casos) | Los brotes (2016, 2023, 2024) generan una cola muy larga; el logaritmo estabiliza la escala. |
| `lluvia`, `temp_max`, `temp_min`, `humedad` | Variables climáticas del enunciado. |
| Rezagos: humedad (1 sem), lluvia (3), temp_min (8), temp_max (12) | Rezago de mayor correlación encontrado en el EDA («el clima actúa semanas después»). |
| `sem_sin`, `sem_cos` | Codifican la semana epidemiológica de forma cíclica (la semana 52 queda cerca de la 1). |

Los rezagos usan `shift()`, es decir, **solo información pasada**. Las primeras 12 semanas se pierden porque no tienen todos los rezagos.

In [ ]:
df_feat = L.crear_features(df)
print("Features del modelo:", L.columnas_features())
print("Filas tras crear rezagos:", len(df_feat), "(se pierden", len(df) - len(df_feat), "semanas iniciales)")
display(df_feat[["fecha_inicio_semana"] + L.columnas_features()].head())

## 4. Escalado, ventanas y partición temporal

**Proceso:** escalar con `MinMaxScaler` → ventanas de 26 semanas → partición temporal.

* **Entrenamiento:** objetivo en 2014–2021 · **Validación:** 2022 · **Prueba:** 2023–2024 (años de brote).
* Los escaladores se ajustan **solo con el entrenamiento** (sin fuga de información).
* Las ventanas cuyo objetivo cruza dos particiones se **descartan**, para que ningún dato futuro contamine el entrenamiento.
* El objetivo es `log(1+casos)` escalado de las 4 semanas siguientes; las métricas se calculan **en casos reales** (se deshace el escalado y el logaritmo).

In [ ]:
datos = L.preparar_datos(df)
m = datos["masks"]
fechas = datos["df_feat"]["fecha_inicio_semana"].to_numpy()

filas = []
for nombre in ["train", "val", "test"]:
    idx = datos["idx_obj"][m[nombre]]
    filas.append({"Partición": nombre, "Ventanas": int(m[nombre].sum()),
                  "Primera semana objetivo": pd.Timestamp(fechas[idx[0]]).date(),
                  "Última semana objetivo": pd.Timestamp(fechas[idx[-1] + L.HORIZONTE - 1]).date()})
print("X:", datos["X"].shape, "(ventanas, 26 semanas, variables) | y:", datos["y"].shape)
display(pd.DataFrame(filas))

## 5. Baselines de referencia

* **Baseline estacional (profesor):** para cada semana objetivo, los casos de la misma semana epidemiológica del año anterior.
* **Persistencia (extra):** repetir los casos de la última semana observada durante las 4 semanas.

Cualquier modelo útil debería, como mínimo, superar el baseline estacional.

In [ ]:
def tabla_baselines(particion):
    mask = m[particion]
    real = L.casos_reales_ventanas(datos, mask)
    base_est, ok = L.baseline_estacional(datos, mask)
    base_per = L.baseline_persistencia(datos, mask)
    return real, {
        "Baseline estacional": (real[ok], base_est[ok]),
        "Persistencia (última semana)": (real, base_per),
    }

filas = []
for part in ["val", "test"]:
    _, pares = tabla_baselines(part)
    for nombre, (r, p) in pares.items():
        filas.append({"Partición": part, "Modelo": nombre, "RMSE": L.rmse(r, p), "MAE": L.mae(r, p)})
resultados_baseline = pd.DataFrame(filas)
display(resultados_baseline)

## 6. Entrenamiento del modelo base: SimpleRNN

`SimpleRNN(32) → Dense(4)`, optimizador Adam, pérdida MSE y *EarlyStopping* sobre la validación (2022).

In [ ]:
n_feat = datos["X"].shape[2]
X_tr, y_tr = datos["X"][m["train"]], datos["y"][m["train"]]
X_va, y_va = datos["X"][m["val"]],   datos["y"][m["val"]]

rnn = L.construir_simplernn(n_feat)
rnn.summary()
hist_rnn = L.entrenar(rnn, X_tr, y_tr, X_va, y_va, epochs=EPOCAS)
print("SimpleRNN — épocas entrenadas:", len(hist_rnn.history["loss"]),
      "| mejor época:", int(np.argmin(hist_rnn.history["val_loss"])) + 1)

## 7. Entrenamiento del modelo principal: LSTM

`LSTM(64) → LSTM(32) → Dropout(0.2) → Dense(4)` (la arquitectura del enunciado).

In [ ]:
lstm = L.construir_lstm(n_feat)
lstm.summary()
hist_lstm = L.entrenar(lstm, X_tr, y_tr, X_va, y_va, epochs=EPOCAS)
mejor_epoca_lstm = int(np.argmin(hist_lstm.history["val_loss"])) + 1
print("LSTM — épocas entrenadas:", len(hist_lstm.history["loss"]), "| mejor época:", mejor_epoca_lstm)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for eje, h, titulo in zip(ejes, [hist_rnn, hist_lstm], ["SimpleRNN (base)", "LSTM (principal)"]):
    eje.plot(h.history["loss"], label="Entrenamiento")
    eje.plot(h.history["val_loss"], label="Validación (2022)")
    eje.set(title=f"Curva de aprendizaje — {titulo}", xlabel="Época", ylabel="MSE (escala normalizada)")
    eje.legend()
plt.tight_layout(); plt.savefig(L.RUTA_FIGURAS / "curvas_aprendizaje.png", dpi=150); plt.show()

## 8. Evaluación: RMSE y MAE (en casos semanales)

Las métricas se calculan sobre las 4 semanas pronosticadas de cada ventana. El **conjunto de prueba (2023–2024)** contiene los dos mayores brotes de la serie, por lo que es una prueba exigente: los valores de 2023 superan el máximo visto en entrenamiento (1 000 casos).

In [ ]:
res = {"SimpleRNN": {}, "LSTM": {}}
for part in ["val", "test"]:
    res["SimpleRNN"][part] = L.evaluar_en(datos, rnn, part)
    res["LSTM"][part] = L.evaluar_en(datos, lstm, part)

filas = []
for part in ["val", "test"]:
    _, pares = tabla_baselines(part)
    for nombre, (r, p) in pares.items():
        filas.append({"Partición": part, "Modelo": nombre, "RMSE": L.rmse(r, p), "MAE": L.mae(r, p)})
    for nombre in ["SimpleRNN", "LSTM"]:
        mt = res[nombre][part]["metricas"]
        filas.append({"Partición": part, "Modelo": nombre, "RMSE": mt["RMSE"], "MAE": mt["MAE"]})

comparacion = pd.DataFrame(filas)
display(comparacion[comparacion["Partición"] == "test"].drop(columns="Partición").reset_index(drop=True))
print("Validación (2022, año de baja actividad):")
display(comparacion[comparacion["Partición"] == "val"].drop(columns="Partición").reset_index(drop=True))

### 8.1 Error por horizonte de pronóstico (semana +1 a +4)

In [ ]:
filas = []
for nombre in ["SimpleRNN", "LSTM"]:
    mt = res[nombre]["test"]["metricas"]
    for h in range(1, L.HORIZONTE + 1):
        filas.append({"Modelo": nombre, "Horizonte": f"+{h} sem", "RMSE": mt[f"RMSE_h{h}"], "MAE": mt[f"MAE_h{h}"]})
display(pd.DataFrame(filas).pivot(index="Horizonte", columns="Modelo", values=["RMSE", "MAE"]))

### 8.2 Predicción vs. casos reales en el conjunto de prueba (2023–2024)

In [ ]:
idx_test = datos["idx_obj"][m["test"]]
fig, ejes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for eje, h in zip(ejes, [0, 3]):
    f = datos["df_feat"]["fecha_inicio_semana"].to_numpy()[idx_test + h]
    eje.plot(f, res["LSTM"]["test"]["real"][:, h], "k", lw=2, label="Casos reales")
    eje.plot(f, res["LSTM"]["test"]["pred"][:, h], label="LSTM")
    eje.plot(f, res["SimpleRNN"]["test"]["pred"][:, h], label="SimpleRNN", alpha=0.8)
    base, ok = L.baseline_estacional(datos, m["test"])
    eje.plot(f[ok], base[ok, h], "--", label="Baseline estacional", alpha=0.7)
    eje.set(title=f"Pronóstico a +{h + 1} semana(s)", ylabel="Casos semanales"); eje.legend()
ejes[1].set_xlabel("Semana pronosticada")
plt.tight_layout(); plt.savefig(L.RUTA_FIGURAS / "prueba_real_vs_predicho.png", dpi=150); plt.show()

## 9. Guardado de los modelos evaluados

Se guardan en `/models` el modelo (`.keras`), los escaladores (`.joblib`) y las métricas (`.json`). Estos archivos los consume la app de Streamlit.

In [ ]:
def info_modelo(nombre):
    return {"modelo": nombre, "ventana_semanas": L.VENTANA, "horizonte_semanas": L.HORIZONTE,
            "features": L.columnas_features(), "particion": "train<=2021 | val=2022 | test=2023-2024",
            "metricas_test": res[nombre]["test"]["metricas"], "metricas_val": res[nombre]["val"]["metricas"]}

L.guardar_artefactos(rnn,  datos["esc_x"], datos["esc_y"], "simplernn_dengue", info_modelo("SimpleRNN"))
L.guardar_artefactos(lstm, datos["esc_x"], datos["esc_y"], "lstm_dengue",      info_modelo("LSTM"))
print("Guardado en:", L.RUTA_MODELOS)

## 10. Modelos finales y pronóstico 2025–2026 (SimpleRNN y LSTM)

**Qué datos existen:** el CSV llega hasta la semana 52 de **2024**. No hay valores reales de 2025 ni 2026 en el dataset, por lo que esos años son **pronósticos**: no se pueden evaluar con RMSE/MAE hasta contar con observaciones reales. La evaluación (RMSE/MAE) se hizo en 2023–2024 (sección 8).

Para pronosticar conviene usar toda la información disponible, incluidos los brotes de 2023–2024. Por eso se reentrena **cada modelo con todas las ventanas**, con el número de épocas óptimo hallado en validación, y los escaladores se reajustan con toda la serie.

**Pronóstico recursivo:** se predicen 4 semanas, se agregan al historial y se repite hasta fin de 2026. El clima de 2025–2026 no existe en el dataset y **no se inventa**: se usa la *climatología* (promedio histórico por semana), un supuesto explícito. Si se consigue clima real o un escenario, `pronosticar_futuro(..., clima_futuro=...)` lo acepta.

⚠️ El modelo está validado a 4 semanas. Más allá, los errores se acumulan y la curva tiende a suavizarse hacia el patrón estacional promedio: es un **escenario orientativo**.

In [ ]:
datos_f = L.preparar_datos(df, ajustar_con_todo=True)
mf = datos_f["masks"]["train"]
n_f = datos_f["X"].shape[2]
epocas_rnn = max(int(np.argmin(hist_rnn.history["val_loss"])) + 1, 20)
epocas_lstm = max(mejor_epoca_lstm, 20)

rnn_final = L.construir_simplernn(n_f)
L.entrenar(rnn_final, datos_f["X"][mf], datos_f["y"][mf], epochs=epocas_rnn)
lstm_final = L.construir_lstm(n_f)
L.entrenar(lstm_final, datos_f["X"][mf], datos_f["y"][mf], epochs=epocas_lstm)

for nombre, mod, ep in [("simplernn_dengue_final", rnn_final, epocas_rnn), ("lstm_dengue_final", lstm_final, epocas_lstm)]:
    L.guardar_artefactos(mod, datos_f["esc_x"], datos_f["esc_y"], nombre,
                         {"modelo": nombre, "entrenado_con": "toda la serie 2014-2024", "epocas": ep,
                          "features": L.columnas_features(), "ventana_semanas": L.VENTANA,
                          "horizonte_semanas": L.HORIZONTE})
print("Modelos finales guardados en", L.RUTA_MODELOS)

In [ ]:
fut = {}
for nombre, mod in [("SimpleRNN", rnn_final), ("LSTM", lstm_final)]:
    fut[nombre] = L.pronosticar_futuro(mod, df, datos_f["esc_x"], datos_f["esc_y"], hasta_anio=2026)

futuro = fut["LSTM"][["fecha_inicio_semana", "anio", "semana_epi"]].copy()
futuro["casos_reales"] = np.nan                      # no existen datos reales de 2025-2026 en el dataset
futuro["pred_SimpleRNN"] = fut["SimpleRNN"]["casos_pred"].round(1).to_numpy()
futuro["pred_LSTM"] = fut["LSTM"]["casos_pred"].round(1).to_numpy()
futuro["periodo"] = "pronostico_" + futuro["anio"].astype(str)
futuro["clima_usado"] = fut["LSTM"]["clima_usado"].to_numpy()
# Suma de la semana y las 3 siguientes (comparable con un umbral endémico en la app)
futuro["casos_acum_4sem_LSTM"] = futuro["pred_LSTM"].rolling(4).sum().shift(-3).round(1)

evaluacion = L.tabla_evaluacion(datos, {"SimpleRNN": res["SimpleRNN"]["test"]["pred"],
                                        "LSTM": res["LSTM"]["test"]["pred"]})
evaluacion[["pred_SimpleRNN", "pred_LSTM"]] = evaluacion[["pred_SimpleRNN", "pred_LSTM"]].round(1)

salida = pd.concat([evaluacion, futuro], ignore_index=True)
L.RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
salida.to_csv(L.RUTA_PROCESADOS / "predicciones_lstm_2025_2026.csv", index=False)
comparacion.to_csv(L.RUTA_PROCESADOS / "metricas_lstm.csv", index=False)
print(salida["periodo"].value_counts().to_dict())
display(futuro.head(5)); display(futuro.tail(3))

In [ ]:
real_anual = df.groupby("anio")["casos_dengue"].sum().rename("Casos reales")
pred_anual = futuro.groupby("anio")[["pred_SimpleRNN", "pred_LSTM"]].sum()
display(pd.concat([real_anual, pred_anual], axis=1).tail(6))

plt.figure(figsize=(15, 5))
rec = df[df["anio"] >= 2021]
plt.plot(rec["fecha_inicio_semana"], rec["casos_dengue"], "k", lw=2, label="Casos reales (observados, hasta 2024)")
plt.plot(evaluacion["fecha_inicio_semana"], evaluacion["pred_LSTM"], color="tab:blue", alpha=0.6, label="LSTM (evaluación 2023–24, +1 sem)")
plt.plot(futuro["fecha_inicio_semana"], futuro["pred_LSTM"], color="tab:red", label="LSTM — pronóstico 2025–2026")
plt.plot(futuro["fecha_inicio_semana"], futuro["pred_SimpleRNN"], color="tab:orange", ls="--", label="SimpleRNN — pronóstico 2025–2026")
plt.axvline(df["fecha_inicio_semana"].max(), color="gray", ls="--")
plt.axvspan(df["fecha_inicio_semana"].max(), futuro["fecha_inicio_semana"].max(), color="gray", alpha=0.08, label="Zona de pronóstico (sin datos reales)")
plt.title("Dengue en Costa Rica: observado, evaluación y pronóstico 2025–2026")
plt.xlabel("Fecha"); plt.ylabel("Casos semanales"); plt.legend(); plt.tight_layout(); plt.savefig(L.RUTA_FIGURAS / "pronostico_2025_2026.png", dpi=150); plt.show()

## 11. Conclusiones

> *Completar con los resultados reales después de ejecutar el notebook.*

* **Comparación de modelos:** ¿LSTM supera a SimpleRNN en RMSE/MAE de prueba? ¿Por cuánto?
* **Frente a los baselines:** ¿supera al baseline estacional? ¿y a la persistencia?
* **Horizonte:** ¿cómo crece el error de +1 a +4 semanas?
* **Limitaciones:** el conjunto de prueba (2023–2024) contiene brotes mayores que todo lo visto en entrenamiento; la validación (2022) es un año de baja actividad; el clima de 2025–2026 se aproximó con la climatología; los pronósticos más allá de 4 semanas son orientativos.
* **Pronóstico 2025–2026:** describir la forma de la curva (estacionalidad) y su incertidumbre.